# 03 · LoRA-DPO persona (Task 5) + all generations (Tasks 2 & 6) + blind pair files
**Settings:** GPU T4 x2 · Internet On · Save & Run All (Commit).

**Needs:** outputs of notebooks **01** and **02** attached.
**Produces:** `outputs/persona_lora/adapter`, `outputs/gen/*.json`, `outputs/eval/*_pairs.jsonl` (~1–1.5 h).

In [1]:
# Code repo (public, or add a GitHub token as a Kaggle secret if private)
REPO_URL = "https://github.com/NifraWahaj/cs5323_PA2.git"
BRANCH = "main"

import os, subprocess, sys
REPO = "/kaggle/working/sprout"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

14109c4 Fix SFT validation OOM on T4; fail notebooks loudly when training fails



In [2]:
# Kaggle's PyTorch already matches the GPU runtime: do NOT reinstall torch.
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 182.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 73.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sproutlm (pyproject.toml) ... done


In [3]:
!python -m pip install -q "trl>=0.12,<0.25" "peft>=0.13" matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 754.0 kB/s eta 0:00:00


In [4]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("GPUs visible:", torch.cuda.device_count())
assert torch.cuda.device_count() == 2, "Set Accelerator to GPU T4 x2"

import trl, peft; print('trl', trl.__version__, '| peft', peft.__version__)

index, name, memory.total [MiB]
0, Tesla T4, 15360 MiB
1, Tesla T4, 15360 MiB
torch 2.11.0+cu128 | transformers 4.57.6 | datasets 4.8.5
GPUs visible: 2
trl 0.24.0 | peft 0.20.0


In [5]:
from sproutlm.kaggle_paths import find_run
SFT = find_run("sft"); DPO_HH = find_run("dpo_hh"); DPO_UF = find_run("dpo_uf")
BASE = "Qwen/Qwen2.5-0.5B"; SMOL = "HuggingFaceTB/SmolLM2-360M-Instruct"
G, E = "/kaggle/working/outputs/gen", "/kaggle/working/outputs/eval"
print(SFT, DPO_HH, DPO_UF, sep="\n")

/kaggle/input/notebooks/nifraw/01notebookgenai/outputs/sft/best
/kaggle/input/notebooks/nifraw/02notebookgenai/outputs/dpo_hh/best
/kaggle/input/notebooks/nifraw/02notebookgenai/outputs/dpo_uf/best


## Task 5 · TRL DPOTrainer + LoRA on the Ramsay pairs (~10 min)

In [6]:
!CUDA_VISIBLE_DEVICES=0 python -m sproutlm.persona_lora --config configs/persona_lora.yaml

[sprout-lora] train 200  val 20
tokenizer_config.json: 3.76kB [00:00, 13.6MB/s]
vocab.json: 801kB [00:00, 44.9MB/s]
merges.txt: 466kB [00:00, 125MB/s]
tokenizer.json: 2.10MB [00:00, 174MB/s]
config.json: 100%|█████████████████████████████| 846/846 [00:00<00:00, 5.28MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100%|██████████████████████| 724M/724M [00:03<00:00, 221MB/s]
generation_config.json: 100%|██████████████████| 132/132 [00:00<00:00, 1.15MB/s]
Extracting prompt in train dataset: 100%|█| 200/200 [00:00<00:00, 3129.41 exampl
Applying chat template to train dataset: 100%|█| 200/200 [00:00<00:00, 1990.34 e
Tokenizing train dataset: 100%|██████| 200/200 [00:00<00:00, 1327.44 examples/s]
Extracting prompt in eval dataset: 100%|█| 20/20 [00:00<00:00, 2803.96 examples/
Applying chat template to eval dataset: 100%|█| 20/20 [00:00<00:00, 1850.16 exam
Tokenizing eval dataset: 100%|█████████| 20/20 [00:00<00:00, 1199.30 examples/s]
[sprout-lora] trainable params

In [7]:
LORA = '/kaggle/working/outputs/persona_lora/adapter'
!cat /kaggle/working/outputs/persona_lora/final_eval.json

{
  "eval_loss": 0.11162008345127106,
  "eval_runtime": 1.1732,
  "eval_samples_per_second": 17.047,
  "eval_steps_per_second": 4.262,
  "eval_rewards/chosen": 1.0114094018936157,
  "eval_rewards/rejected": -1.2734687328338623,
  "eval_rewards/accuracies": 1.0,
  "eval_rewards/margins": 2.2848782539367676,
  "eval_logps/chosen": -156.24069213867188,
  "eval_logps/rejected": -135.16026306152344,
  "eval_logits/chosen": 0.2972581386566162,
  "eval_logits/rejected": 0.355047345161438,
  "epoch": 3.0
}

## Sanity check on 3 prompts per model before the full runs (checkpoint path, template, EOS, length)

In [8]:
!python -m sproutlm.generate --model {BASE}   --eval_set alpaca  --generator base    --limit 3
!python -m sproutlm.generate --model {SFT}    --eval_set alpaca  --generator sft     --limit 3
!python -m sproutlm.generate --model {DPO_UF} --eval_set alpaca  --generator dpo_uf  --limit 3
!python -m sproutlm.generate --model {DPO_HH} --eval_set hh      --generator dpo_hh  --limit 3
!python -m sproutlm.generate --model {SMOL} --adapter {LORA} --style chat_template --eval_set persona --generator smol_lora --limit 3

tokenizer_config.json: 7.23kB [00:00, 27.9MB/s]
vocab.json: 2.78MB [00:00, 99.6MB/s]
merges.txt: 1.67MB [00:00, 147MB/s]
tokenizer.json: 7.03MB [00:00, 200MB/s]
config.json: 100%|█████████████████████████████| 681/681 [00:00<00:00, 2.67MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100%|██████████████████████| 988M/988M [00:04<00:00, 241MB/s]
generation_config.json: 100%|███████████████████| 138/138 [00:00<00:00, 978kB/s]
  3/3
[alpaca-42-000] Create a short, concise summary of the paper based on its abstract.

Few-shot learning (FSL) is one of the key future steps in machine learning and raises a lot of attention. In this paper, we focus on the FSL problem of dialogue understanding, which contains two closely related tasks: intent detection and slot filling. Dialogue understanding has been proven to benefit a lot from jointly learning the two sub-tasks. However, such joint learning becomes challenging in the few-shot scenarios: on the one hand, the sparsity

## Full generation: two queues in parallel, one per GPU
Greedy, `max_new_tokens=384`, `repetition_penalty=1.1` for **every** model (see `sproutlm/generate.py: DECODE`).
Each file is written as soon as its model finishes; existing files are skipped, so a rerun resumes.

In [9]:
q0 = f"""
python -m sproutlm.generate --device cuda:0 --model {BASE}   --eval_set alpaca --generator base   --out {G}/alpaca__base.json
python -m sproutlm.generate --device cuda:0 --model {SFT}    --eval_set alpaca --generator sft    --out {G}/alpaca__sft.json
python -m sproutlm.generate --device cuda:0 --model {DPO_UF} --eval_set alpaca --generator dpo_uf --out {G}/alpaca__dpo_uf.json
"""
q1 = f"""
python -m sproutlm.generate --device cuda:1 --model {SFT}    --eval_set hh --generator sft    --out {G}/hh__sft.json
python -m sproutlm.generate --device cuda:1 --model {DPO_HH} --eval_set hh --generator dpo_hh --out {G}/hh__dpo_hh.json
python -m sproutlm.generate --device cuda:1 --model {BASE}   --eval_set hh --generator base   --out {G}/hh__base.json
python -m sproutlm.generate --device cuda:1 --model {SMOL} --style chat_template --eval_set persona --generator smol_base --out {G}/persona__smol_base.json
python -m sproutlm.generate --device cuda:1 --model {SMOL} --adapter {LORA} --style chat_template --eval_set persona --generator smol_lora --out {G}/persona__smol_lora.json
"""
import subprocess, os
os.makedirs(G, exist_ok=True)
procs = [subprocess.Popen(["bash", "-c", "set -e;" + q.strip().replace("\n", " && ")],
                          stdout=open(f"{G}/queue{i}.log", "w"), stderr=subprocess.STDOUT)
         for i, q in enumerate([q0, q1])]
codes = [p.wait() for p in procs]
print("exit codes:", codes)
!tail -n 3 {G}/queue0.log {G}/queue1.log
assert codes == [0, 0], "a generation queue failed; see the logs above"

exit codes: [0, 0]
==> /kaggle/working/outputs/gen/queue0.log <==
  288/300
  300/300
[sprout-gen] wrote /kaggle/working/outputs/gen/alpaca__dpo_uf.json (300 rows, 4.25 min)

==> /kaggle/working/outputs/gen/queue1.log <==
  16/24
  24/24
[sprout-gen] wrote /kaggle/working/outputs/gen/persona__smol_lora.json (24 rows, 0.58 min)


## Blind pairwise files (A/B order randomised per prompt; model identity kept in hidden metadata)

In [10]:
!python -m sproutlm.pairs --candidate {G}/alpaca__dpo_uf.json   --baseline {G}/alpaca__sft.json      --rubric general         --out {E}/dpo_uf_vs_sft_pairs.jsonl
!python -m sproutlm.pairs --candidate {G}/hh__dpo_hh.json       --baseline {G}/hh__sft.json          --rubric safety          --out {E}/dpo_hh_vs_sft_pairs.jsonl
!python -m sproutlm.pairs --candidate {G}/persona__smol_lora.json --baseline {G}/persona__smol_base.json --rubric persona_style   --out {E}/lora_vs_base_style_pairs.jsonl
!python -m sproutlm.pairs --candidate {G}/persona__smol_lora.json --baseline {G}/persona__smol_base.json --rubric persona_quality --out {E}/lora_vs_base_quality_pairs.jsonl
!python -m sproutlm.pairs --candidate {G}/alpaca__sft.json      --baseline {G}/alpaca__base.json     --rubric general         --out {E}/extra_sft_vs_base_pairs.jsonl

[sprout-pairs] /kaggle/working/outputs/eval/dpo_uf_vs_sft_pairs.jsonl: 300 pairs, candidate shown as A in 146
[sprout-pairs] /kaggle/working/outputs/eval/dpo_hh_vs_sft_pairs.jsonl: 300 pairs, candidate shown as A in 157
[sprout-pairs] /kaggle/working/outputs/eval/lora_vs_base_style_pairs.jsonl: 24 pairs, candidate shown as A in 12
[sprout-pairs] /kaggle/working/outputs/eval/lora_vs_base_quality_pairs.jsonl: 24 pairs, candidate shown as A in 12
[sprout-pairs] /kaggle/working/outputs/eval/extra_sft_vs_base_pairs.jsonl: 300 pairs, candidate shown as A in 146


In [11]:
# Ship the judge code with the outputs so notebook 04 works even without Internet.
!mkdir -p /kaggle/working/outputs/code && cp -r sproutlm scripts /kaggle/working/outputs/code/
import json
for line in open(f"{E}/dpo_uf_vs_sft_pairs.jsonl").readlines()[:1]:
    print(json.dumps(json.loads(line), indent=1)[:1500])

{
 "pair_id": "alpaca-42-000",
 "instruction": "Create a short, concise summary of the paper based on its abstract.\n\nFew-shot learning (FSL) is one of the key future steps in machine learning and raises a lot of attention. In this paper, we focus on the FSL problem of dialogue understanding, which contains two closely related tasks: intent detection and slot filling. Dialogue understanding has been proven to benefit a lot from jointly learning the two sub-tasks. However, such joint learning becomes challenging in the few-shot scenarios: on the one hand, the sparsity of samples greatly magnifies the difficulty of modeling the connection between the two tasks; on the other hand, how to jointly learn multiple tasks in the few-shot setting is still less investigated. In response to this, we introduce FewJoint, the first FSL benchmark for joint dialogue understanding. FewJoint provides a new corpus with 59 different dialogue domains from real industrial API and a code platform to ease FSL

In [12]:
# Archive this stage (easier to download/attach), then: Save Version -> "Save & Run All (Commit)".
!cd /kaggle/working && zip -qr generation_outputs.zip outputs/gen outputs/eval outputs/persona_lora/adapter outputs/persona_lora/*.json* && ls -lh /kaggle/working && du -sh /kaggle/working/outputs/*

total 34M
-rw-r--r--  1 root root  34M Oct 15 13:14 generation_outputs.zip
-rw-r--r--  1 root root 131K Oct 15 13:14 __notebook__.ipynb
drwxr-xr-x  6 root root 4.0K Oct 15 13:14 outputs
drwxr-xr-x 13 root root 4.0K Oct 15 12:56 sprout
172K	/kaggle/working/outputs/code
2.1M	/kaggle/working/outputs/eval
2.5M	/kaggle/working/outputs/gen
38M	/kaggle/working/outputs/persona_lora
